In [1]:
# System
import os
import sys

os.environ["KERAS_BACKEND"] = "jax"
sys.path.append("../..")

In [2]:
# Setup
import json
from importlib import import_module
from pathlib import Path

import numpy as np
from keras import ops
from rich.table import Table

from src.models import GradientBoostedDecisionTree as BDT
from src.models import LearnableCutFlowParallel as LCF_PAR
from src.models import LearnableCutFlowSequential as LCF_SEQ
from src.models import MultiLayerPerceptron as MLP
from src.utils import Timer, load_model, print, to_numpy

In [3]:
# Parameters
rerun = False
n_runs = 10

# Dataset
dataset = "real1"  # *
selected_feature_indices = [0, 3, 4]  # *
n_samples = 200000
seed = 42


# Model
centers = [80, 0.15, 0.025, 2, 2, 0.3]  # *
features = [
    r"$M_{jet}$",
    r"$C_2^{\beta=1}$",
    r"$C_2^{\beta=2}$",
    r"$D_2^{\beta=1}$",
    r"$D_2^{\beta=2}$",
    r"$\tau_{21}^{\beta=1}$",
]  # *

n_epochs = 200
batch_size = 512
verbose = 0

if not rerun and Path("results.json").exists():
    with open("results.json", "r") as f:
        results = json.load(f)
else:
    results = {}

In [4]:
# Dataset *
module = import_module(f"src.datasets.{dataset}")
load_data = getattr(module, "load_data")
(x_train, y_train), (x_test, y_test) = load_data(n_samples, seed)

x_train = x_train[:, selected_feature_indices]
x_test = x_test[:, selected_feature_indices]
centers = [centers[i] for i in selected_feature_indices]
features = [features[i] for i in selected_feature_indices]

selection = np.ones_like(x_train[:, 0], dtype=bool)
for i in range(x_train.shape[1]):
    p05 = np.percentile(x_train[:, i], 5)
    p95 = np.percentile(x_train[:, i], 95)
    selection = (p05 < x_train[:, i]) & (x_train[:, i] < p95) & selection

x_train = x_train[selection]
y_train = y_train[selection]

selection = np.ones_like(x_test[:, 0], dtype=bool)
for i in range(x_test.shape[1]):
    p05 = np.percentile(x_test[:, i], 5)
    p95 = np.percentile(x_test[:, i], 95)
    selection = (p05 < x_test[:, i]) & (x_test[:, i] < p95) & selection

x_test = x_test[selection]
y_test = y_test[selection]

print(f"{x_train.shape=}")
print(f"{y_train.shape=}")
print(f"{x_test.shape=}")
print(f"{y_test.shape=}")

x_train.shape=(83329, 3)
y_train.shape=(83329, 1)
x_test.shape=(83268, 3)
y_test.shape=(83268, 1)


In [5]:
# Model: BDT
bdt_name = "bdt"
bdt_ckpt_paths = [Path(f"checkpoints/{bdt_name}@{i + 1}.pkl") for i in range(n_runs)]

bdt_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in bdt_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {bdt_name}@{i + 1}...")

        bdt = BDT(input_shape=x_train.shape, name=bdt_name)
        bdt.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            bdt.fit(
                x_train,
                y_train.squeeze(),
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        bdt.save(bdt_ckpt_paths[i])
        bdt_ckpts.append(load_model(bdt_ckpt_paths[i]))

    results[bdt_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[bdt_name]['training_time_mean']:.2f} ± "
    f"{results[bdt_name]['training_time_std']:.2f} seconds"
)

KeyError: 'training_time_mean'

In [6]:
# Model: MLP
mlp_name = "mlp"
mlp_ckpt_paths = [Path(f"checkpoints/{mlp_name}@{i + 1}.keras") for i in range(n_runs)]

mlp_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in mlp_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {mlp_name}@{i + 1}...")

        mlp = MLP(x_train.shape, name=mlp_name)
        mlp.adapt(x_train)
        mlp.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            mlp.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        mlp.save(mlp_ckpt_paths[i])
        mlp_ckpts.append(load_model(mlp_ckpt_paths[i]))

    results[mlp_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[mlp_name]['training_time_mean']:.2f} ± "
    f"{results[mlp_name]['training_time_std']:.2f} seconds"
)

Processing mlp@1...
Epoch 1/200
163/163 - 11s - 66ms/step - loss: 0.4748
Epoch 2/200
163/163 - 1s - 7ms/step - loss: 0.3430
Epoch 3/200
163/163 - 0s - 952us/step - loss: 0.3397
Epoch 4/200
163/163 - 0s - 974us/step - loss: 0.3383
Epoch 5/200
163/163 - 0s - 1ms/step - loss: 0.3373
Epoch 6/200
163/163 - 0s - 999us/step - loss: 0.3369
Epoch 7/200
163/163 - 0s - 1ms/step - loss: 0.3365
Epoch 8/200
163/163 - 0s - 1ms/step - loss: 0.3356
Epoch 9/200
163/163 - 0s - 1ms/step - loss: 0.3354
Epoch 10/200
163/163 - 0s - 1ms/step - loss: 0.3355
Epoch 11/200
163/163 - 0s - 1ms/step - loss: 0.3345
Epoch 12/200
163/163 - 0s - 1ms/step - loss: 0.3343
Epoch 13/200
163/163 - 0s - 1ms/step - loss: 0.3337
Epoch 14/200
163/163 - 0s - 1ms/step - loss: 0.3332
Epoch 15/200
163/163 - 0s - 1ms/step - loss: 0.3331
Epoch 16/200
163/163 - 0s - 1ms/step - loss: 0.3334
Epoch 17/200
163/163 - 0s - 932us/step - loss: 0.3331
Epoch 18/200
163/163 - 0s - 1ms/step - loss: 0.3322
Epoch 19/200
163/163 - 0s - 970us/step - lo

In [7]:
# Model: LCF(parallel)
lcf_par_name = "lcf_par"
lcf_par_ckpt_paths = [
    Path(f"checkpoints/{lcf_par_name}@{i + 1}.keras") for i in range(n_runs)
]

lcf_par_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in lcf_par_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {lcf_par_name}@{i + 1}...")

        lcf_par = LCF_PAR(x_train.shape, centers, features=features, name=lcf_par_name)
        lcf_par.adapt(x_train)
        lcf_par.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            lcf_par.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        lcf_par.save(lcf_par_ckpt_paths[i])
        lcf_par_ckpts.append(load_model(lcf_par_ckpt_paths[i]))

    results[lcf_par_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[lcf_par_name]['training_time_mean']:.2f} ± "
    f"{results[lcf_par_name]['training_time_std']:.2f} seconds"
)

Processing lcf_par@1...
Epoch 1/200
163/163 - 3s - 21ms/step - loss: 0.3513
Epoch 2/200
163/163 - 1s - 7ms/step - loss: 0.3309
Epoch 3/200
163/163 - 0s - 1ms/step - loss: 0.3154
Epoch 4/200
163/163 - 0s - 1ms/step - loss: 0.3040
Epoch 5/200
163/163 - 0s - 1ms/step - loss: 0.2958
Epoch 6/200
163/163 - 0s - 1ms/step - loss: 0.2898
Epoch 7/200
163/163 - 0s - 1ms/step - loss: 0.2854
Epoch 8/200
163/163 - 0s - 1ms/step - loss: 0.2821
Epoch 9/200
163/163 - 0s - 1ms/step - loss: 0.2796
Epoch 10/200
163/163 - 0s - 1ms/step - loss: 0.2775
Epoch 11/200
163/163 - 0s - 2ms/step - loss: 0.2758
Epoch 12/200
163/163 - 0s - 1ms/step - loss: 0.2744
Epoch 13/200
163/163 - 0s - 1ms/step - loss: 0.2732
Epoch 14/200
163/163 - 0s - 1ms/step - loss: 0.2721
Epoch 15/200
163/163 - 0s - 1ms/step - loss: 0.2712
Epoch 16/200
163/163 - 0s - 1ms/step - loss: 0.2703
Epoch 17/200
163/163 - 0s - 1ms/step - loss: 0.2694
Epoch 18/200
163/163 - 0s - 1ms/step - loss: 0.2685
Epoch 19/200
163/163 - 0s - 1ms/step - loss: 0.2

In [8]:
# Model: LCF(sequential)
lcf_seq_name = "lcf_seq"
lcf_seq_ckpt_paths = [
    Path(f"checkpoints/{lcf_seq_name}@{i + 1}.keras") for i in range(n_runs)
]

lcf_seq_ckpts = []
if rerun or not all(ckpt.exists() for ckpt in lcf_seq_ckpt_paths):
    records = []
    for i in range(n_runs):
        print(f"Processing {lcf_seq_name}@{i + 1}...")

        lcf_seq = LCF_SEQ(x_train.shape, centers, features=features, name=lcf_seq_name)
        lcf_seq.adapt(x_train)
        lcf_seq.compile(optimizer="adam", loss="crossentropy")

        with Timer() as timer:
            lcf_seq.fit(
                x_train,
                y_train,
                batch_size=batch_size,
                epochs=n_epochs,
                verbose=verbose,
            )
        records.append(timer.record)

        lcf_seq.save(lcf_seq_ckpt_paths[i])
        lcf_seq_ckpts.append(load_model(lcf_seq_ckpt_paths[i]))

    results[lcf_seq_name] = {
        "training_time_mean": np.mean(records),
        "training_time_std": np.std(records),
    }

print(
    "Training time: "
    f"{results[lcf_seq_name]['training_time_mean']:.2f} ± "
    f"{results[lcf_seq_name]['training_time_std']:.2f} seconds"
)

Processing lcf_seq@1...
Epoch 1/200
163/163 - 4s - 27ms/step - loss: 0.1159
Epoch 2/200
163/163 - 1s - 7ms/step - loss: 0.1149
Epoch 3/200
163/163 - 0s - 1ms/step - loss: 0.1244
Epoch 4/200
163/163 - 0s - 1ms/step - loss: 0.1309
Epoch 5/200
163/163 - 0s - 1ms/step - loss: 0.1313
Epoch 6/200
163/163 - 0s - 1ms/step - loss: 0.1298
Epoch 7/200
163/163 - 0s - 1ms/step - loss: 0.1282
Epoch 8/200
163/163 - 0s - 1ms/step - loss: 0.1268
Epoch 9/200
163/163 - 0s - 1ms/step - loss: 0.1256
Epoch 10/200
163/163 - 0s - 1ms/step - loss: 0.1245
Epoch 11/200
163/163 - 0s - 1ms/step - loss: 0.1238
Epoch 12/200
163/163 - 0s - 2ms/step - loss: 0.1233
Epoch 13/200
163/163 - 0s - 1ms/step - loss: 0.1229
Epoch 14/200
163/163 - 0s - 1ms/step - loss: 0.1225
Epoch 15/200
163/163 - 0s - 1ms/step - loss: 0.1221
Epoch 16/200
163/163 - 0s - 1ms/step - loss: 0.1219
Epoch 17/200
163/163 - 0s - 1ms/step - loss: 0.1216
Epoch 18/200
163/163 - 0s - 1ms/step - loss: 0.1213
Epoch 19/200
163/163 - 0s - 1ms/step - loss: 0.1

In [10]:
# Analysis: metrics
rerun = True
y_true = y_test

table = Table(title="Model Performance Comparison")
table.add_column("#", justify="center", style="cyan", no_wrap=True)
table.add_column("Model", style="magenta")
table.add_column("TP", justify="right", style="green")
table.add_column("FP", justify="right", style="red")
table.add_column("Accuracy", justify="right", style="blue")
table.add_column("Precision", justify="right", style="blue")
table.add_column("Significance", justify="right", style="yellow")
table.add_column("Time(s)", justify="right", style="yellow")

if rerun or not Path("resultsx10.json").exists():
    for ckpts in [bdt_ckpts, mlp_ckpts, lcf_par_ckpts, lcf_seq_ckpts]:
        print(f"Processing {ckpts[0].name}...")

        tp_list = []
        fp_list = []
        accuracy_list = []
        precision_list = []
        significance_list = []

        for i, ckpt in enumerate(ckpts):
            y_pred = ckpt.predict(x_test, batch_size=batch_size, verbose=0)
            y_pred = ops.all(y_pred > 0.5, axis=1, keepdims=True)

            tp = to_numpy(ops.sum((y_true == 1) & (y_pred == 1)))
            fp = to_numpy(ops.sum((y_true == 0) & (y_pred == 1)))
            tn = to_numpy(ops.sum((y_true == 0) & (y_pred == 0)))
            fn = to_numpy(ops.sum((y_true == 1) & (y_pred == 0)))

            accuracy = (tp + tn) / (tp + tn + fp + fn)
            precision = tp / (tp + fp)

            s = tp / (tp + tn + fp + fn) * 3000 * 1000 * 0.7644
            b = fp / (tp + tn + fp + fn) * 3000 * 1000 * 1.806 * 1e5
            significance = s / np.sqrt(b)

            tp_list.append(tp)
            fp_list.append(fp)
            accuracy_list.append(accuracy)
            precision_list.append(precision)
            significance_list.append(significance)

        tp_mean = np.mean(tp_list)
        fp_mean = np.mean(fp_list)
        accuracy_mean = np.mean(accuracy_list)
        precision_mean = np.mean(precision_list)
        significance_mean = np.mean(significance_list)

        tp_std = np.std(tp_list)
        fp_std = np.std(fp_list)
        accuracy_std = np.std(accuracy_list)
        precision_std = np.std(precision_list)
        significance_std = np.std(significance_list)

        results[ckpts[0].name].update(
            {
                "tp_mean": tp_mean.tolist(),
                "tp_std": tp_std.tolist(),
                "fp_mean": fp_mean.tolist(),
                "fp_std": fp_std.tolist(),
                "accuracy_mean": accuracy_mean.tolist(),
                "accuracy_std": accuracy_std.tolist(),
                "precision_mean": precision_mean.tolist(),
                "precision_std": precision_std.tolist(),
                "significance_mean": significance_mean.tolist(),
                "significance_std": significance_std.tolist(),
            }
        )

        with open("resultsx10.json", "w") as f:
            json.dump(results, f, indent=4)

for i, (name, metrics) in enumerate(results.items()):
    tp_mean = metrics["tp_mean"]
    tp_std = metrics["tp_std"]
    fp_mean = metrics["fp_mean"]
    fp_std = metrics["fp_std"]
    accuracy_mean = metrics["accuracy_mean"]
    accuracy_std = metrics["accuracy_std"]
    precision_mean = metrics["precision_mean"]
    precision_std = metrics["precision_std"]
    significance_mean = metrics["significance_mean"]
    significance_std = metrics["significance_std"]
    training_time_mean = metrics["training_time_mean"]
    training_time_std = metrics["training_time_std"]

    table.add_row(
        str(i + 1),
        name,
        f"{tp_mean:.0f}\n± {tp_std:.0f}",
        f"{fp_mean:.0f}\n± {fp_std:.0f}",
        f"{accuracy_mean:.4f}\n± {accuracy_std:.4f}",
        f"{precision_mean:.4f}\n± {precision_std:.4f}",
        f"{significance_mean:.4f}\n± {significance_std:.4f}",
        f"{training_time_mean:.2f}\n± {training_time_std:.2f}",
    )

print(table)

Processing bdt...


Processing mlp...
Processing lcf_par...
Processing lcf_seq...
                         Model Performance Comparison                          
┏━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ # ┃ Model   ┃    TP ┃    FP ┃ Accuracy ┃ Precision ┃ Significance ┃ Time(s) ┃
┡━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ 1 │ bdt     │ 41828 │  7232 │   0.8553 │    0.8526 │       5.3101 │   12.88 │
│   │         │   ± 0 │   ± 0 │ ± 0.0000 │  ± 0.0000 │     ± 0.0002 │  ± 0.08 │
│ 2 │ mlp     │ 42249 │  7327 │   0.8592 │    0.8523 │       5.3329 │   58.86 │
│   │         │ ± 360 │ ± 375 │ ± 0.0007 │  ± 0.0054 │     ± 0.0903 │  ± 4.65 │
│ 3 │ lcf_par │ 34086 │  4014 │   0.8010 │    0.8947 │       5.8091 │   52.59 │
│   │         │  ± 51 │  ± 34 │ ± 0.0002 │  ± 0.0007 │     ± 0.0158 │  ± 1.18 │
│ 4 │ lcf_seq │ 39028 │  6323 │   0.8326 │    0.8606 │       5.2989 │   55.59 │
│   │         │  ± 19 │  ± 20 │ ± 0.0001 │  ± 0.0003 │    